In [ ]:
"""
===============================================================================
Script Name: 01_two_mass_ode_training.py
Description: Trains a 2-Mass Physics-Informed ODE to calibrate thermal 
             parameters. Models Silicon Die (C_die) and Heatsink/Chassis 
             (C_sink) interaction via Thermal Paste (R_paste).
             Optimized for 10,000-segment chunked data. Features persistent 
             GPU-RAM validation, double-shuffled training, dynamic fan curves, 
             bulletproof memory cleanup, and automated learning plots.
===============================================================================
"""

import os
import sys
import time
import warnings
import json
import subprocess
import threading
import random
import gc
from pathlib import Path
from datetime import timedelta
import queue

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# --- 1. CONFIGURATION ---
warnings.filterwarnings('ignore')

ENABLE_PREFETCH = True # Set to True to mask SSD latency since CPU RAM is abundant

# Hyperparameters
EPOCHS = 500
LEARNING_RATE = 0.01
DT = 0.11
VAL_BATCH_SIZE = 10000 # To prevent OOM during validation's 39k segment forward pass

# --- 2. SETUP UTILITIES ---
def select_gpu():
    """
    Selects GPU via CLI arg, env var, or interactive prompt.
    Includes live status print via nvidia-smi.
    """
    # 1. Check CLI Arguments
    for i, arg in enumerate(sys.argv):
        if arg == "--gpu" and i + 1 < len(sys.argv):
            gpu_id = int(sys.argv[i + 1])
            print(f"[SYSTEM] GPU {gpu_id} selected via CLI.")
            return gpu_id

    # 2. Check Environment Variables
    env_gpu = os.environ.get("GPU_ID")
    if env_gpu is not None and env_gpu.isdigit():
        print(f"[SYSTEM] GPU {env_gpu} selected via Environment Var.")
        return int(env_gpu)

    # 3. Print nvidia-smi status before asking for input
    print("\n[SYSTEM] Live GPU Status:")
    try:
        # We use run() here for a one-time print to terminal
        subprocess.run(["nvidia-smi"], check=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        print("[WARNING] nvidia-smi not found or failed. Cannot display GPU stats.")

    # 4. Fallback to interactive input
    if not torch.cuda.is_available():
        print("[WARNING] No CUDA detected. Using CPU.")
        return None
        
    while True:
        gpu_id = input("\n[SYSTEM] Enter GPU ID (e.g., 0, 1) [Enter for '0']: ").strip()
        if not gpu_id: return 0
        if gpu_id.isdigit() and int(gpu_id) < torch.cuda.device_count():
            return int(gpu_id)

class DualLogger:
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.log = open(filepath, "a", encoding="utf-8")
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()
    def flush(self):
        self.terminal.flush()
        self.log.flush()
    def close(self):
        self.log.close()

class ChunkPrefetcher:
    def __init__(self, chunk_files):
        self.chunk_files = chunk_files
        self._next_chunk = None
        self._thread = None
    def _load_in_background(self, filepath):
        self._next_chunk = torch.load(filepath, map_location='cpu', weights_only=False)
    def prefetch(self, chunk_idx):
        if chunk_idx < len(self.chunk_files):
            self._thread = threading.Thread(target=self._load_in_background, args=(self.chunk_files[chunk_idx],))
            self._thread.start()
    def get(self):
        if self._thread is not None:
            self._thread.join()
            self._thread = None
        chunk = self._next_chunk
        self._next_chunk = None
        return chunk

# --- 3. THE 2-MASS PHYSICS MODEL (ODE) ---
class ThermalODEModel(nn.Module):
    def __init__(self):
        super(ThermalODEModel, self).__init__()
        
        prior_C_die     = 2.0
        prior_C_sink    = 500.0
        prior_R_paste   = 0.05
        prior_k01       = 0.0780
        prior_k10       = 0.0281
        prior_q         = 0.0
        prior_h_base    = 4.0
        prior_h_active  = 12.0
        prior_T_thresh  = 65.0
        prior_beta      = 0.5
        
        self.bounds = {
            'C_die_0':    (0.1, 10.0),      'C_die_1':    (0.1, 10.0),
            'C_sink_0':   (100.0, 5000.0),  'C_sink_1':   (100.0, 5000.0),
            'R_paste_0':  (0.001, 0.2),     'R_paste_1':  (0.001, 0.2),
            'k01':        (0.0, 0.5),       'k10':        (0.0, 0.5),
            'q0':         (-10.0, 10.0),    'q1':         (-10.0, 10.0),
            'h_base_0':   (0.5, 10.0),      'h_base_1':   (0.5, 10.0),
            'h_active_0': (0.0, 30.0),      'h_active_1': (0.0, 30.0),
            'T_thresh_0': (40.0, 85.0),     'T_thresh_1': (40.0, 85.0),
            'beta_0':     (0.05, 2.0),      'beta_1':     (0.05, 2.0)
        }
        
        def inv_sigmoid(val, low, high):
            norm = (val - low) / (high - low)
            norm = max(min(norm, 0.999), 0.001)
            return torch.log(torch.tensor(norm / (1.0 - norm)))

        self.raw_C_die_0   = nn.Parameter(inv_sigmoid(prior_C_die, *self.bounds['C_die_0']))
        self.raw_C_die_1   = nn.Parameter(inv_sigmoid(prior_C_die, *self.bounds['C_die_1']))
        self.raw_C_sink_0  = nn.Parameter(inv_sigmoid(prior_C_sink, *self.bounds['C_sink_0']))
        self.raw_C_sink_1  = nn.Parameter(inv_sigmoid(prior_C_sink, *self.bounds['C_sink_1']))
        self.raw_R_paste_0 = nn.Parameter(inv_sigmoid(prior_R_paste, *self.bounds['R_paste_0']))
        self.raw_R_paste_1 = nn.Parameter(inv_sigmoid(prior_R_paste, *self.bounds['R_paste_1']))

        self.raw_k01       = nn.Parameter(inv_sigmoid(prior_k01, *self.bounds['k01']))
        self.raw_k10       = nn.Parameter(inv_sigmoid(prior_k10, *self.bounds['k10']))
        self.raw_q0        = nn.Parameter(inv_sigmoid(prior_q, *self.bounds['q0']))
        self.raw_q1        = nn.Parameter(inv_sigmoid(prior_q, *self.bounds['q1']))
        
        self.raw_h_base_0   = nn.Parameter(inv_sigmoid(prior_h_base, *self.bounds['h_base_0']))
        self.raw_h_active_0 = nn.Parameter(inv_sigmoid(prior_h_active, *self.bounds['h_active_0']))
        self.raw_T_thresh_0 = nn.Parameter(inv_sigmoid(prior_T_thresh, *self.bounds['T_thresh_0']))
        self.raw_beta_0     = nn.Parameter(inv_sigmoid(prior_beta, *self.bounds['beta_0']))
        
        self.raw_h_base_1   = nn.Parameter(inv_sigmoid(prior_h_base, *self.bounds['h_base_1']))
        self.raw_h_active_1 = nn.Parameter(inv_sigmoid(prior_h_active, *self.bounds['h_active_1']))
        self.raw_T_thresh_1 = nn.Parameter(inv_sigmoid(prior_T_thresh, *self.bounds['T_thresh_1']))
        self.raw_beta_1     = nn.Parameter(inv_sigmoid(prior_beta, *self.bounds['beta_1']))

    def get_physical_params(self):
        p = {}
        for name, bound in self.bounds.items():
            raw_val = getattr(self, f"raw_{name}")
            low, high = bound
            p[name] = low + torch.sigmoid(raw_val) * (high - low)
        return p

    def forward(self, P0, P1, T0_init, T1_init, T_amb):
        params = self.get_physical_params()
        batch_size, seq_len = P0.shape
        
        T0_preds = torch.empty((batch_size, seq_len), device=P0.device)
        T1_preds = torch.empty((batch_size, seq_len), device=P1.device)
        
        T0_die, T1_die = T0_init, T1_init
        T0_sink = T0_die - (P0[:, 0] * params['R_paste_0'])
        T1_sink = T1_die - (P1[:, 0] * params['R_paste_1'])
        
        T0_preds[:, 0] = T0_die
        T1_preds[:, 0] = T1_die
        
        inv_C_die0, inv_C_die1 = 1.0 / params['C_die_0'], 1.0 / params['C_die_1']
        inv_C_sink0, inv_C_sink1 = 1.0 / params['C_sink_0'], 1.0 / params['C_sink_1']
        
        for t in range(seq_len - 1):
            h0_curr = params['h_base_0'] + params['h_active_0'] * torch.sigmoid(params['beta_0'] * (T0_die - params['T_thresh_0']))
            h1_curr = params['h_base_1'] + params['h_active_1'] * torch.sigmoid(params['beta_1'] * (T1_die - params['T_thresh_1']))
            
            dT0_die = (P0[:, t] - (T0_die - T0_sink) / params['R_paste_0']) * inv_C_die0
            dT1_die = (P1[:, t] - (T1_die - T1_sink) / params['R_paste_1']) * inv_C_die1
            
            dT0_sink = ((T0_die - T0_sink) / params['R_paste_0'] + params['k01'] * P1[:, t] - h0_curr * (T0_sink - T_amb) + params['q0']) * inv_C_sink0
            dT1_sink = ((T1_die - T1_sink) / params['R_paste_1'] + params['k10'] * P0[:, t] - h1_curr * (T1_sink - T_amb) + params['q1']) * inv_C_sink1
            
            T0_die = T0_die + DT * dT0_die
            T1_die = T1_die + DT * dT1_die
            T0_sink = T0_sink + DT * dT0_sink
            T1_sink = T1_sink + DT * dT1_sink
            
            T0_preds[:, t+1] = T0_die
            T1_preds[:, t+1] = T1_die
            
        return T0_preds, T1_preds

# --- 4. MAIN EXECUTION ---
def main():
    total_start_time = time.perf_counter()
    global script_name
    try:
        script_path = Path(__file__).resolve()
        script_name = script_path.stem
    except NameError:
        script_name = "01_two_mass_ode_training"
        current_dir = Path.cwd() 
        script_path = current_dir / f"{script_name}.py"
        
    project_root = script_path.parent.parent.parent
    home_dir = Path.home()
    
    data_dir = project_root / "Implementation" / "data" / "mit-supercloud-dataset" / "gpu" / "dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_tensors"
    
    user_prefix = input("\nEnter version prefix (e.g., v1_base) [Press Enter for default 'v1']: ").strip()
    VERSION_PREFIX = user_prefix if user_prefix else "v1"
    
    SAVE_DIR = project_root / "Implementation" / "models" / VERSION_PREFIX
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    PLOTS_DIR = SAVE_DIR / "training_plots"
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)

    gpu_id = select_gpu()
    DEVICE = torch.device(f"cuda:{gpu_id}" if gpu_id is not None else "cpu")

    log_path = SAVE_DIR / f"{script_name}_terminal_output.txt"
    sys.stdout = DualLogger(log_path)

    print("=" * 70)
    print(f"--- STARTING: {script_name.upper()} ---")
    print(f"[*] Version Prefix: {VERSION_PREFIX}")
    print(f"[*] Device: {DEVICE}")
    print(f"[*] Prefetching: {'Enabled' if ENABLE_PREFETCH else 'Disabled'}")
    print(f"[*] Saving to: {str(SAVE_DIR).replace(str(home_dir), '~')}")
    print("=" * 70)

    hist_train_rmse, hist_val_rmse, hist_lr = [], [], []
    hist_Csink0, hist_Csink1 = [], []
    hist_hact0, hist_hact1 = [], []

    gpu_logger_process = None
    if gpu_id is not None:
        gpu_csv_path = SAVE_DIR / f"{VERSION_PREFIX}_gpu{gpu_id}_timeseries.csv"
        try:
            smi_cmd = ["nvidia-smi", "--query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,temperature.gpu,power.draw", "--format=csv", "-i", str(gpu_id), "-l", "5", "-f", str(gpu_csv_path)]
            gpu_logger_process = subprocess.Popen(smi_cmd)
            print(f"[*] Started continuous GPU logging to: {gpu_csv_path.name}")
        except FileNotFoundError:
            pass

    try:
        model = ThermalODEModel().to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
        
        mse_criterion = nn.MSELoss(reduction='none') 
        mae_criterion = nn.L1Loss(reduction='none')  

        RESUME_PATH = SAVE_DIR / f"{VERSION_PREFIX}_resume_checkpoint.pt"
        BEST_PATH = SAVE_DIR / f"{VERSION_PREFIX}_best_model.pt"
        CSV_LOG_PATH = SAVE_DIR / "training_log.csv"

        start_epoch = 1
        best_val_loss = float('inf')

        if RESUME_PATH.exists():
            print(f"\n[SYSTEM] Found checkpoint at: {RESUME_PATH.name}. Resuming...")
            checkpoint = torch.load(RESUME_PATH, map_location=DEVICE, weights_only=False)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            scheduler.load_state_dict(checkpoint.get('scheduler_state_dict', scheduler.state_dict()))
            start_epoch = checkpoint['epoch'] + 1
            best_val_loss = checkpoint['best_val_loss']
            torch.set_rng_state(checkpoint['torch_rng'].cpu() if isinstance(checkpoint['torch_rng'], torch.Tensor) else checkpoint['torch_rng'])
            np.random.set_state(checkpoint['numpy_rng'])
        else:
            with open(CSV_LOG_PATH, "w") as f:
                f.write("epoch,train_rmse,val_rmse,train_mae,val_mae,lr,C_die0,C_sink0,R_pst0,C_die1,C_sink1,R_pst1,k01,k10,h_base0,h_act0,T_thr0,h_base1,h_act1,T_thr1,q0,q1,epoch_time_sec\n")

        train_chunks = sorted(list(data_dir.glob("train_chunk_*.pt")))
        val_chunks = sorted(list(data_dir.glob("val_chunk*.pt")))
        print(f"Found {len(train_chunks)} Train chunk(s) and {len(val_chunks)} Val chunk(s).")

        # PERMANENT GPU ALLOCATION FOR VALIDATION
        print(f"\n[SYSTEM] Pre-loading Validation bundle permanently into GPU VRAM...")
        val_data_cpu = torch.load(val_chunks[0], map_location='cpu', weights_only=False)
        val_data = {
            'P0': val_data_cpu['P0'].to(DEVICE),
            'P1': val_data_cpu['P1'].to(DEVICE),
            'T0': val_data_cpu['T0'].to(DEVICE),
            'T1': val_data_cpu['T1'].to(DEVICE),
            'T_amb': val_data_cpu['T_amb'].to(DEVICE),
            'valid_len': val_data_cpu['valid_len'].to(DEVICE)
        }
        del val_data_cpu
        # gc.collect()

        for epoch in range(start_epoch, EPOCHS + 1):
            epoch_start_time = time.perf_counter()
            
            # --- 1. TRAINING PHASE ---
            model.train()
            train_mse_accum, train_mae_accum, train_steps = 0.0, 0.0, 0
            
            # MACRO-SHUFFLE: Randomize the order of the chunk files each epoch
            epoch_train_chunks = train_chunks.copy()
            random.shuffle(epoch_train_chunks)
            
            if ENABLE_PREFETCH:
                prefetcher = ChunkPrefetcher(epoch_train_chunks)
                prefetcher.prefetch(0)
            
            # PROGRESS BAR added directly to the chunk iterator
            chunk_iterator = tqdm(range(len(epoch_train_chunks)), desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=False)
            
            for chunk_idx in chunk_iterator:
                if ENABLE_PREFETCH:
                    chunk_data = prefetcher.get()
                    prefetcher.prefetch(chunk_idx + 1)
                else:
                    chunk_data = torch.load(epoch_train_chunks[chunk_idx], map_location='cpu', weights_only=False)
                
                # MICRO-SHUFFLE: Randomize the indices inside this specific 10k chunk
                total_size = chunk_data['P0'].shape[0]
                indices = torch.randperm(total_size)
                
                # Push the full, shuffled 10k chunk directly to GPU
                P0 = chunk_data['P0'][indices].to(DEVICE, non_blocking=True)
                P1 = chunk_data['P1'][indices].to(DEVICE, non_blocking=True)
                T0_true = chunk_data['T0'][indices].to(DEVICE, non_blocking=True)
                T1_true = chunk_data['T1'][indices].to(DEVICE, non_blocking=True)
                T_amb = chunk_data['T_amb'][indices].to(DEVICE, non_blocking=True)
                valid_len = chunk_data['valid_len'][indices].to(DEVICE, non_blocking=True)
                
                optimizer.zero_grad()
                
                T0_init, T1_init = T0_true[:, 0], T1_true[:, 0]
                T0_pred, T1_pred = model(P0, P1, T0_init, T1_init, T_amb)
                
                seq_len = P0.shape[1]
                mask = torch.arange(seq_len, device=DEVICE).expand(len(valid_len), seq_len) < valid_len.unsqueeze(1)
                
                mse_0 = mse_criterion(T0_pred, T0_true)
                mse_1 = mse_criterion(T1_pred, T1_true)
                masked_mse = ((mse_0 + mse_1) * mask).sum() / (2 * mask.sum())
                
                with torch.no_grad():
                    mae_0 = mae_criterion(T0_pred.detach(), T0_true)
                    mae_1 = mae_criterion(T1_pred.detach(), T1_true)
                    masked_mae = ((mae_0 + mae_1) * mask).sum() / (2 * mask.sum())

                masked_mse.backward()
                optimizer.step()
                
                train_mse_accum += masked_mse.item()
                train_mae_accum += masked_mae.item()
                train_steps += 1
                
                # VRAM WIPE (Train Phase) - Free the GPU and CPU tensors
                del P0, P1, T0_true, T1_true, T_amb, valid_len, T0_pred, T1_pred, mask
                del mse_0, mse_1, masked_mse, mae_0, mae_1, masked_mae
                del chunk_data 
                # gc.collect()
                # torch.cuda.empty_cache()
            
            train_rmse = (train_mse_accum / train_steps) ** 0.5
            train_mae = train_mae_accum / train_steps
            
            # --- 2. VALIDATION PHASE ---
            model.eval()
            val_mse_accum, val_mae_accum, val_steps = 0.0, 0.0, 0
            
            with torch.no_grad():
                total_val_size = val_data['P0'].shape[0]
                
                # Iterate through the permanently stored GPU tensors in chunks to prevent OOM
                for start_idx in range(0, total_val_size, VAL_BATCH_SIZE):
                    end_idx = min(start_idx + VAL_BATCH_SIZE, total_val_size)
                    
                    # No transfer overhead. Just slicing the VRAM tensors.
                    P0_val = val_data['P0'][start_idx:end_idx]
                    P1_val = val_data['P1'][start_idx:end_idx]
                    T0_true_val = val_data['T0'][start_idx:end_idx]
                    T1_true_val = val_data['T1'][start_idx:end_idx]
                    T_amb_val = val_data['T_amb'][start_idx:end_idx]
                    valid_len_val = val_data['valid_len'][start_idx:end_idx]
                    
                    T0_init_v, T1_init_v = T0_true_val[:, 0], T1_true_val[:, 0]
                    T0_pred_v, T1_pred_v = model(P0_val, P1_val, T0_init_v, T1_init_v, T_amb_val)
                    
                    seq_len_v = P0_val.shape[1]
                    mask_v = torch.arange(seq_len_v, device=DEVICE).expand(len(valid_len_val), seq_len_v) < valid_len_val.unsqueeze(1)
                    
                    mse_0_v = mse_criterion(T0_pred_v, T0_true_val)
                    mse_1_v = mse_criterion(T1_pred_v, T1_true_val)
                    val_mse = ((mse_0_v + mse_1_v) * mask_v).sum() / (2 * mask_v.sum())
                    
                    mae_0_v = mae_criterion(T0_pred_v, T0_true_val)
                    mae_1_v = mae_criterion(T1_pred_v, T1_true_val)
                    val_mae_t = ((mae_0_v + mae_1_v) * mask_v).sum() / (2 * mask_v.sum())
                    
                    val_mse_accum += val_mse.item()
                    val_mae_accum += val_mae_t.item()
                    val_steps += 1
                    
                    # VRAM WIPE (Val Phase) - Free the intermediate graph tensors
                    del P0_val, P1_val, T0_true_val, T1_true_val, T_amb_val, valid_len_val
                    del T0_pred_v, T1_pred_v, mask_v, mse_0_v, mse_1_v, val_mse, mae_0_v, mae_1_v, val_mae_t
                
            val_rmse = (val_mse_accum / val_steps) ** 0.5
            val_mae = val_mae_accum / val_steps
            
            # gc.collect()
            # torch.cuda.empty_cache()
                
            epoch_time = time.perf_counter() - epoch_start_time
            current_lr = optimizer.param_groups[0]['lr']
            p = {k: v.item() for k, v in model.get_physical_params().items()}
            
            # --- 3. LOGGING ---
            hist_train_rmse.append(train_rmse)
            hist_val_rmse.append(val_rmse)
            hist_lr.append(current_lr)
            hist_Csink0.append(p['C_sink_0'])
            hist_Csink1.append(p['C_sink_1'])
            hist_hact0.append(p['h_active_0'])
            hist_hact1.append(p['h_active_1'])

            print(f"\n--- Epoch {epoch}/{EPOCHS} ---")
            print(f"Time: {epoch_time:.1f}s | Train RMSE: {train_rmse:.4f} °C | train MAE: {train_mae:.4f} °C | Val RMSE: {val_rmse:.4f} °C | Val MAE: {val_mae:.4f} °C | LR: {current_lr:.6f}")
            
            is_best = val_rmse < best_val_loss
            if is_best:
                best_val_loss = val_rmse
                print(">>> New best parameters saved! <<<")
                
            print(f"[Param] k01: {p['k01']:.4f} | k10: {p['k10']:.4f} | q0: {p['q0']:+.2f} | q1: {p['q1']:+.2f}")
            print(f"[GPU 0] C_die: {p['C_die_0']:5.2f} | C_sink: {p['C_sink_0']:6.1f} | R_pst: {p['R_paste_0']:.4f} | h_base: {p['h_base_0']:.2f} | h_act: {p['h_active_0']:5.2f} | T_thr: {p['T_thresh_0']:.1f}")
            print(f"[GPU 1] C_die: {p['C_die_1']:5.2f} | C_sink: {p['C_sink_1']:6.1f} | R_pst: {p['R_paste_1']:.4f} | h_base: {p['h_base_1']:.2f} | h_act: {p['h_active_1']:5.2f} | T_thr: {p['T_thresh_1']:.1f}")

            scheduler.step(val_rmse)
            
            with open(CSV_LOG_PATH, "a") as f:
                f.write(f"{epoch},{train_rmse:.6f},{val_rmse:.6f},{train_mae:.6f},{val_mae:.6f},{current_lr:.6f},"
                        f"{p['C_die_0']:.4f},{p['C_sink_0']:.4f},{p['R_paste_0']:.4f},"
                        f"{p['C_die_1']:.4f},{p['C_sink_1']:.4f},{p['R_paste_1']:.4f},"
                        f"{p['k01']:.4f},{p['k10']:.4f},{p['h_base_0']:.4f},{p['h_active_0']:.4f},{p['T_thresh_0']:.4f},"
                        f"{p['h_base_1']:.4f},{p['h_active_1']:.4f},{p['T_thresh_1']:.4f},{p['q0']:.4f},{p['q1']:.4f},{epoch_time:.2f}\n")
    
            checkpoint_state = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_loss': best_val_loss,
                'torch_rng': torch.get_rng_state(),
                'numpy_rng': np.random.get_state()
            }
            torch.save(checkpoint_state, RESUME_PATH)
            
            if is_best:
                torch.save(checkpoint_state, BEST_PATH)
                with open(SAVE_DIR / "calibrated_physics.json", "w") as f:
                    json.dump(p, f, indent=4)

    except KeyboardInterrupt:
        print("\n[!] Training interrupted by user (Ctrl+C).")
    except Exception as e:
        print(f"\n[!] Fatal Error during training: {e}")
    finally:
        print("\n[*] Initializing strict RAM/VRAM cleanup & Plot Generation...")
        
        if gpu_logger_process is not None:
            gpu_logger_process.terminate()
            gpu_logger_process.wait()
            
        for var in ['P0', 'P1', 'T0_true', 'T1_true', 'T_amb', 'valid_len', 'val_data', 'chunk_data', 'epoch_train_chunks']:
            if var in locals():
                del locals()[var]
        
        # gc.collect()
        # if torch.cuda.is_available():
            # torch.cuda.empty_cache()
            
        print("[*] Memory cleared successfully.")
        
        if len(hist_train_rmse) > 0:
            epochs_arr = range(1, len(hist_train_rmse) + 1)
            
            plt.figure(figsize=(10, 6))
            plt.plot(epochs_arr, hist_train_rmse, label='Train RMSE', color='blue', linewidth=2)
            plt.plot(epochs_arr, hist_val_rmse, label='Val RMSE', color='red', linewidth=2, linestyle='--')
            plt.title("Learning Curves (RMSE vs Epoch)")
            plt.xlabel("Epoch")
            plt.ylabel("RMSE (°C)")
            plt.legend()
            plt.grid(True)
            plt.savefig(PLOTS_DIR / "learning_curves.png")
            plt.close()
            
            plt.figure(figsize=(10, 6))
            plt.plot(epochs_arr, hist_Csink0, label='C_sink GPU 0', color='purple', linewidth=2)
            plt.plot(epochs_arr, hist_Csink1, label='C_sink GPU 1', color='orange', linewidth=2)
            plt.title("Heatsink Thermal Mass Evolution (C_sink)")
            plt.xlabel("Epoch")
            plt.ylabel("Thermal Mass (J/°C)")
            plt.legend()
            plt.grid(True)
            plt.savefig(PLOTS_DIR / "thermal_mass_trajectory.png")
            plt.close()
            
            plt.figure(figsize=(10, 6))
            plt.plot(epochs_arr, hist_hact0, label='h_active GPU 0', color='green', linewidth=2)
            plt.plot(epochs_arr, hist_hact1, label='h_active GPU 1', color='brown', linewidth=2)
            plt.title("Active Cooling Fan Coefficient Evolution (h_active)")
            plt.xlabel("Epoch")
            plt.ylabel("Cooling Rate (W/°C)")
            plt.legend()
            plt.grid(True)
            plt.savefig(PLOTS_DIR / "fan_curve_evolution.png")
            plt.close()
            
            print(f"[*] Training plots saved to: {PLOTS_DIR.name}/")

        total_time = time.perf_counter() - total_start_time
        formatted_time = str(timedelta(seconds=int(total_time)))
        print(f"\n[*] Total Training Time: {formatted_time}")
        print("=" * 70)

        if hasattr(sys.stdout, 'terminal'):
            sys.stdout.close()
            sys.stdout = sys.stdout.terminal

if __name__ == "__main__":
    main()
    


Enter version prefix (e.g., v1_base) [Press Enter for default 'v1']:  



[SYSTEM] Live GPU Status:
Sat Apr  4 18:11:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4500               Off |   00000000:3B:00.0 Off |                  Off |
| 31%   60C    P5             23W /  200W |    7870MiB /  20470MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------


[SYSTEM] Enter GPU ID (e.g., 0, 1) [Enter for '0']:  1


--- STARTING: 01_TWO_MASS_ODE_TRAINING ---
[*] Version Prefix: v1
[*] Device: cuda:1
[*] Prefetching: Enabled
[*] Saving to: ~/Capstone/Implementation/models/v1
[*] Started continuous GPU logging to: v1_gpu1_timeseries.csv

[SYSTEM] Found checkpoint at: v1_resume_checkpoint.pt. Resuming...
Found 35 Train chunk(s) and 1 Val chunk(s).

[SYSTEM] Pre-loading Validation bundle permanently into GPU VRAM...



--- Epoch 4/500 ---
Time: 595.2s | Train RMSE: 4.0616 °C | train MAE: 2.9304 °C | Val RMSE: 3.8997 °C | Val MAE: 2.9557 °C | LR: 0.010000
>>> New best parameters saved! <<<
[Param] k01: 0.0570 | k10: 0.0187 | q0: -2.41 | q1: -2.60
[GPU 0] C_die:  3.10 | C_sink:  843.3 | R_pst: 0.0370 | h_base: 4.70 | h_act:  9.36 | T_thr: 69.8
[GPU 1] C_die:  3.15 | C_sink:  822.4 | R_pst: 0.0347 | h_base: 4.99 | h_act: 12.56 | T_thr: 65.9


Epoch 5/500 [Train]:   9%|▊         | 3/35 [00:50<08:52, 16.65s/it]

In [3]:
"""
===============================================================================
Script Name: 02_two_mass_ode_testing.py
Description: Evaluates the trained 2-Mass Physics-Informed ODE on the unseen
             test set. 
             Pre-loads all 671 test files directly into GPU VRAM for maximum
             inference speed. Computes extreme tail-risk metrics, global 
             statistical robustness, generates static plots, and builds 
             interactive Plotly HTMLs for deep diagnostics.
===============================================================================
"""

import os
import sys
import time
import warnings
import subprocess
import gc
from pathlib import Path
from datetime import timedelta

import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm

import matplotlib
matplotlib.use('Agg') # Safe for background rendering without GUI
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# --- 1. PHYSICS MODEL (Must match training exactly to load weights) ---
class ThermalODEModel(nn.Module):
    def __init__(self):
        super(ThermalODEModel, self).__init__()
        
        prior_C_die      = 2.0
        prior_C_sink     = 500.0
        prior_R_paste    = 0.05
        prior_k01        = 0.0780
        prior_k10        = 0.0281
        prior_q          = 0.0
        prior_h_base     = 4.0
        prior_h_active   = 12.0
        prior_T_thresh   = 65.0
        prior_beta       = 0.5
        
        self.bounds = {
            'C_die_0':    (0.1, 10.0),      'C_die_1':    (0.1, 10.0),
            'C_sink_0':   (100.0, 5000.0),  'C_sink_1':   (100.0, 5000.0),
            'R_paste_0':  (0.001, 0.2),     'R_paste_1':  (0.001, 0.2),
            'k01':        (0.0, 0.5),       'k10':        (0.0, 0.5),
            'q0':         (-10.0, 10.0),    'q1':         (-10.0, 10.0),
            'h_base_0':   (0.5, 10.0),      'h_base_1':   (0.5, 10.0),
            'h_active_0': (0.0, 30.0),      'h_active_1': (0.0, 30.0),
            'T_thresh_0': (40.0, 85.0),     'T_thresh_1': (40.0, 85.0),
            'beta_0':     (0.05, 2.0),      'beta_1':     (0.05, 2.0)
        }
        
        def inv_sigmoid(val, low, high):
            norm = (val - low) / (high - low)
            norm = max(min(norm, 0.999), 0.001)
            return torch.log(torch.tensor(norm / (1.0 - norm)))

        self.raw_C_die_0   = nn.Parameter(inv_sigmoid(prior_C_die, *self.bounds['C_die_0']))
        self.raw_C_die_1   = nn.Parameter(inv_sigmoid(prior_C_die, *self.bounds['C_die_1']))
        self.raw_C_sink_0  = nn.Parameter(inv_sigmoid(prior_C_sink, *self.bounds['C_sink_0']))
        self.raw_C_sink_1  = nn.Parameter(inv_sigmoid(prior_C_sink, *self.bounds['C_sink_1']))
        self.raw_R_paste_0 = nn.Parameter(inv_sigmoid(prior_R_paste, *self.bounds['R_paste_0']))
        self.raw_R_paste_1 = nn.Parameter(inv_sigmoid(prior_R_paste, *self.bounds['R_paste_1']))

        self.raw_k01       = nn.Parameter(inv_sigmoid(prior_k01, *self.bounds['k01']))
        self.raw_k10       = nn.Parameter(inv_sigmoid(prior_k10, *self.bounds['k10']))
        self.raw_q0        = nn.Parameter(inv_sigmoid(prior_q, *self.bounds['q0']))
        self.raw_q1        = nn.Parameter(inv_sigmoid(prior_q, *self.bounds['q1']))
        
        self.raw_h_base_0   = nn.Parameter(inv_sigmoid(prior_h_base, *self.bounds['h_base_0']))
        self.raw_h_active_0 = nn.Parameter(inv_sigmoid(prior_h_active, *self.bounds['h_active_0']))
        self.raw_T_thresh_0 = nn.Parameter(inv_sigmoid(prior_T_thresh, *self.bounds['T_thresh_0']))
        self.raw_beta_0     = nn.Parameter(inv_sigmoid(prior_beta, *self.bounds['beta_0']))
        
        self.raw_h_base_1   = nn.Parameter(inv_sigmoid(prior_h_base, *self.bounds['h_base_1']))
        self.raw_h_active_1 = nn.Parameter(inv_sigmoid(prior_h_active, *self.bounds['h_active_1']))
        self.raw_T_thresh_1 = nn.Parameter(inv_sigmoid(prior_T_thresh, *self.bounds['T_thresh_1']))
        self.raw_beta_1     = nn.Parameter(inv_sigmoid(prior_beta, *self.bounds['beta_1']))

    def get_physical_params(self):
        p = {}
        for name, bound in self.bounds.items():
            raw_val = getattr(self, f"raw_{name}")
            low, high = bound
            p[name] = low + torch.sigmoid(raw_val) * (high - low)
        return p

    def forward(self, P0, P1, T0_init, T1_init, T_amb):
        params = self.get_physical_params()
        batch_size, seq_len = P0.shape
        DT = 0.11
        
        T0_preds = torch.empty((batch_size, seq_len), device=P0.device)
        T1_preds = torch.empty((batch_size, seq_len), device=P1.device)
        
        T0_die, T1_die = T0_init, T1_init
        T0_sink = T0_die - (P0[:, 0] * params['R_paste_0'])
        T1_sink = T1_die - (P1[:, 0] * params['R_paste_1'])
        
        T0_preds[:, 0] = T0_die
        T1_preds[:, 0] = T1_die
        
        inv_C_die0, inv_C_die1 = 1.0 / params['C_die_0'], 1.0 / params['C_die_1']
        inv_C_sink0, inv_C_sink1 = 1.0 / params['C_sink_0'], 1.0 / params['C_sink_1']
        
        for t in range(seq_len - 1):
            h0_curr = params['h_base_0'] + params['h_active_0'] * torch.sigmoid(params['beta_0'] * (T0_die - params['T_thresh_0']))
            h1_curr = params['h_base_1'] + params['h_active_1'] * torch.sigmoid(params['beta_1'] * (T1_die - params['T_thresh_1']))
            
            dT0_die = (P0[:, t] - (T0_die - T0_sink) / params['R_paste_0']) * inv_C_die0
            dT1_die = (P1[:, t] - (T1_die - T1_sink) / params['R_paste_1']) * inv_C_die1
            
            dT0_sink = ((T0_die - T0_sink) / params['R_paste_0'] + params['k01'] * P1[:, t] - h0_curr * (T0_sink - T_amb) + params['q0']) * inv_C_sink0
            dT1_sink = ((T1_die - T1_sink) / params['R_paste_1'] + params['k10'] * P0[:, t] - h1_curr * (T1_sink - T_amb) + params['q1']) * inv_C_sink1
            
            T0_die = T0_die + DT * dT0_die
            T1_die = T1_die + DT * dT1_die
            T0_sink = T0_sink + DT * dT0_sink
            T1_sink = T1_sink + DT * dT1_sink
            
            T0_preds[:, t+1] = T0_die
            T1_preds[:, t+1] = T1_die
            
        return T0_preds, T1_preds

# --- 2. LOGGING & SETUP UTILITIES ---
class DualLogger:
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.log = open(filepath, "w", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

    def close(self):
        self.log.close()

def select_gpu():
    for i, arg in enumerate(sys.argv):
        if arg == "--gpu" and i + 1 < len(sys.argv):
            return int(sys.argv[i + 1])
    env_gpu = os.environ.get("GPU_ID")
    if env_gpu is not None and env_gpu.isdigit():
        return int(env_gpu)
    if not torch.cuda.is_available():
        return None
    while True:
        gpu_id = input("\n[SYSTEM] Enter the GPU ID you want to allocate [Press Enter for '0']: ").strip()
        if not gpu_id: return 0
        if gpu_id.isdigit() and int(gpu_id) < torch.cuda.device_count():
            return int(gpu_id)

def calc_rmse(true, pred): return np.sqrt(np.mean((true - pred)**2))
def calc_mae(true, pred): return np.mean(np.abs(true - pred))

# --- 3. PLOTLY GENERATOR ---
def generate_interactive_plot(job_list, stitched_data, filename, title_prefix):
    fig = make_subplots(rows=1, cols=2, subplot_titles=("GPU 0 Temperature", "GPU 1 Temperature"), horizontal_spacing=0.05)
    traces_per_job = 4 
    buttons = []

    for i, run_id in enumerate(job_list):
        job = stitched_data[run_id]
        t = np.arange(len(job['T0_true']))
        visible = (i == 0)
        
        fig.add_trace(go.Scatter(x=t, y=job['T0_true'], mode='lines', line=dict(color='blue'), name="True T0", visible=visible), row=1, col=1)
        fig.add_trace(go.Scatter(x=t, y=job['T0_pred'], mode='lines', line=dict(color='red', dash='dot'), name="Pred T0", visible=visible), row=1, col=1)
        fig.add_trace(go.Scatter(x=t, y=job['T1_true'], mode='lines', line=dict(color='blue'), name="True T1", visible=visible), row=1, col=2)
        fig.add_trace(go.Scatter(x=t, y=job['T1_pred'], mode='lines', line=dict(color='red', dash='dot'), name="Pred T1", visible=visible), row=1, col=2)

        visibility_array = [False] * (len(job_list) * traces_per_job)
        visibility_array[i*traces_per_job : (i+1)*traces_per_job] = [True, True, True, True]
        
        buttons.append(dict(
            label=f"{run_id} (RMSE: {job['rmse']:.2f}°C)",
            method="update",
            args=[{"visible": visibility_array}, {"title": f"{title_prefix} - Run ID: {run_id}"}]
        ))
        
    fig.update_layout(
        title=f"{title_prefix} - Run ID: {job_list[0]}",
        updatemenus=[dict(active=0, buttons=buttons, x=0.5, y=1.15, xanchor="center", yanchor="top")],
        template="plotly_white", hovermode="x unified", height=600
    )
    fig.update_yaxes(title_text="Temperature (°C)", range=[20, 100], row=1, col=1)
    fig.update_yaxes(title_text="Temperature (°C)", range=[20, 100], row=1, col=2)
    fig.write_html(filename)

# --- 4. MAIN SCRIPT ---
def main():
    total_start_time = time.perf_counter()
    try:
        script_path = Path(__file__).resolve()
        script_name = script_path.stem
    except NameError:
        script_name = "02_two_mass_ode_testing"
        script_path = Path.cwd() / f"{script_name}.py"
        
    project_root = script_path.parent.parent.parent
    home_dir = Path.home()
    
    gpu_id = select_gpu()
    DEVICE = torch.device(f"cuda:{gpu_id}" if gpu_id is not None else "cpu")

    user_prefix = input("\nEnter version prefix to load (e.g., v1_base) [Press Enter for default 'v1']: ").strip()
    VERSION_PREFIX = user_prefix if user_prefix else "v1"

    data_dir = project_root / "Implementation" / "data" / "mit-supercloud-dataset" / "gpu" / "dual_gpu_0000_parquet_to_0019_parquet_cleaned_split_tensors"
    models_dir = project_root / "Implementation" / "models" / VERSION_PREFIX
    PLOTS_DIR = models_dir / "testing_plots"
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    
    log_path = models_dir / f"{script_name}_terminal_output.txt"
    sys.stdout = DualLogger(log_path)
    
    BEST_PATH = models_dir / f"{VERSION_PREFIX}_best_model.pt"
    if not BEST_PATH.exists():
        print(f"[!] Could not find best model at {BEST_PATH}. Exiting.")
        if hasattr(sys.stdout, 'terminal'):
            sys.stdout.close()
            sys.stdout = sys.stdout.terminal
        return

    print("=" * 70)
    print(f"--- STARTING: {script_name.upper()} ---")
    print(f"[*] Device: {DEVICE}")
    print(f"[*] Output Directory: {str(PLOTS_DIR).replace(str(home_dir), '~')}")
    print("=" * 70)

    try:
        # 1. Load Model
        model = ThermalODEModel().to(DEVICE)
        checkpoint = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        print(f"[*] Model weights loaded from Epoch {checkpoint['epoch']} (Val RMSE: {checkpoint['best_val_loss']:.4f})")

        # 2. Pre-Load Phase
        test_files = sorted(list(data_dir.glob("test_*.pt")))
        print(f"[*] Pre-loading {len(test_files)} distinct physical test files into GPU VRAM (~3.4 GB)...")
        
        loaded_test_data = {}
        for file in tqdm(test_files, desc="Caching to GPU"):
            run_id = file.stem.replace('test_', '')
            data = torch.load(file, map_location='cpu', weights_only=False)
            
            loaded_test_data[run_id] = {
                'P0': data['P0'].to(DEVICE, non_blocking=True),
                'P1': data['P1'].to(DEVICE, non_blocking=True),
                'T0_true': data['T0'].to(DEVICE, non_blocking=True),
                'T1_true': data['T1'].to(DEVICE, non_blocking=True),
                'T_amb': data['T_amb'].to(DEVICE, non_blocking=True),
                'valid_len': data['valid_len'] # Kept on CPU for integer slicing
            }
            del data
        
        # gc.collect()
        
        # 3. Sequential Inference (Compute Phase)
        jobs_dict = {}
        print("\n[*] Starting pure silicon inference loop...")
        
        with torch.no_grad():
            for run_id, tensors in tqdm(loaded_test_data.items(), desc="Evaluating Physical Jobs"):
                
                T0_init, T1_init = tensors['T0_true'][:, 0], tensors['T1_true'][:, 0]
                T0_pred, T1_pred = model(tensors['P0'], tensors['P1'], T0_init, T1_init, tensors['T_amb'])
                
                # Flatten and slice exact lengths
                t0_t_flat, t1_t_flat = [], []
                t0_p_flat, t1_p_flat = [], []
                
                for i in range(tensors['P0'].shape[0]):
                    v_len = tensors['valid_len'][i].item()
                    t0_t_flat.extend(tensors['T0_true'][i, :v_len].cpu().numpy())
                    t1_t_flat.extend(tensors['T1_true'][i, :v_len].cpu().numpy())
                    t0_p_flat.extend(T0_pred[i, :v_len].cpu().numpy())
                    t1_p_flat.extend(T1_pred[i, :v_len].cpu().numpy())
                    
                job_true = np.concatenate([t0_t_flat, t1_t_flat])
                job_pred = np.concatenate([t0_p_flat, t1_p_flat])
                
                jobs_dict[run_id] = {
                    'T0_true': np.array(t0_t_flat), 'T0_pred': np.array(t0_p_flat),
                    'T1_true': np.array(t1_t_flat), 'T1_pred': np.array(t1_p_flat),
                    'rmse': calc_rmse(job_true, job_pred),
                    'length': len(t0_t_flat)
                }
                
                del T0_pred, T1_pred
                
        # Free VRAM after inference
        del loaded_test_data
        # gc.collect()
        # torch.cuda.empty_cache()
                
        # 4. Global Statistical & Tail-Risk Evaluation
        all_true, all_pred = [], []
        first_10_rmses, last_10_rmses = [], []

        for job in jobs_dict.values():
            all_true.extend(job['T0_true'])
            all_true.extend(job['T1_true'])
            all_pred.extend(job['T0_pred'])
            all_pred.extend(job['T1_pred'])
            
            idx_10 = max(1, int(job['length'] * 0.10))
            first_true = np.concatenate([job['T0_true'][:idx_10], job['T1_true'][:idx_10]])
            first_pred = np.concatenate([job['T0_pred'][:idx_10], job['T1_pred'][:idx_10]])
            last_true = np.concatenate([job['T0_true'][-idx_10:], job['T1_true'][-idx_10:]])
            last_pred = np.concatenate([job['T0_pred'][-idx_10:], job['T1_pred'][-idx_10:]])
            
            first_10_rmses.append(calc_rmse(first_true, first_pred))
            last_10_rmses.append(calc_rmse(last_true, last_pred))

        all_true, all_pred = np.array(all_true), np.array(all_pred)
        abs_errors = np.abs(all_true - all_pred)
        
        global_rmse = calc_rmse(all_true, all_pred)
        global_mae = calc_mae(all_true, all_pred)
        mean_err = np.mean(all_pred - all_true)
        
        p95, p99 = np.percentile(abs_errors, 95), np.percentile(abs_errors, 99)
        max_err = np.max(abs_errors)
        
        danger_mask = all_true > 70.0
        danger_rmse = calc_rmse(all_true[danger_mask], all_pred[danger_mask]) if np.any(danger_mask) else float('nan')

        print("\n" + "=" * 70)
        print("=== FINAL TEST EVALUATION METRICS ===")
        print("=" * 70)
        print(f"Total Physical Jobs Evaluated   : {len(jobs_dict)}")
        print(f"Total Timesteps Evaluated       : {len(all_true):,}")
        print("-" * 70)
        print(f"[Baseline] Global RMSE          : {global_rmse:.4f} °C")
        print(f"[Baseline] Global MAE           : {global_mae:.4f} °C")
        print(f"[Baseline] Directional Bias     : {mean_err:+.4f} °C (>0 means model predicts hotter)")
        print("-" * 70)
        print(f"[Tail-Risk] 95th Percentile Err : {p95:.4f} °C")
        print(f"[Tail-Risk] 99th Percentile Err : {p99:.4f} °C")
        print(f"[Tail-Risk] Absolute Max Error  : {max_err:.4f} °C")
        print("-" * 70)
        print(f"[Extremes] RMSE in Danger Zone  : {danger_rmse:.4f} °C (True Temp > 70°C)")
        print("-" * 70)
        print(f"[Stability] First 10% RMSE      : {np.mean(first_10_rmses):.4f} °C")
        print(f"[Stability] Last 10% RMSE       : {np.mean(last_10_rmses):.4f} °C")
        print("=" * 70)

        # 5. Global Static Plots
        print("\n[*] Generating Global Static Plots...")
        sns.set_theme(style="whitegrid")
        
        # 5a. Error Distribution
        plt.figure(figsize=(10, 6))
        sns.histplot(abs_errors, bins=100, kde=True, color="purple")
        plt.xlim(0, p99 * 1.5)
        plt.title("Global Absolute Error Distribution")
        plt.xlabel("Absolute Error (°C)")
        plt.ylabel("Frequency")
        plt.savefig(PLOTS_DIR / "global_error_distribution.png", dpi=150)
        plt.close()

        # 5b. Hexbin Heatmap
        plt.figure(figsize=(8, 8))
        plt.hexbin(all_true, all_pred, gridsize=60, cmap='inferno', mincnt=1)
        plt.plot([20, 90], [20, 90], 'w--', linewidth=2, label="Ideal (y=x)")
        plt.title("Actual vs Predicted Temperatures (Heatmap)")
        plt.xlabel("True Temperature (°C)")
        plt.ylabel("Predicted Temperature (°C)")
        plt.colorbar(label='Count in Bin')
        plt.legend()
        plt.savefig(PLOTS_DIR / "actual_vs_predicted_heatmap.png", dpi=150)
        plt.close()

        # 5c. Temporal Drift Boxplots
        plt.figure(figsize=(8, 6))
        sns.boxplot(data=[first_10_rmses, last_10_rmses], palette="Set2")
        plt.xticks([0, 1], ['First 10% of Job', 'Last 10% of Job'])
        plt.ylabel("RMSE (°C)")
        plt.title("Temporal Integration Stability")
        plt.savefig(PLOTS_DIR / "temporal_stability_boxplots.png", dpi=150)
        plt.close()

        # 6. Interactive HTMLs
        print("[*] Generating Interactive Plotly HTML Diagnostics...")
        sorted_job_ids = sorted(jobs_dict.keys(), key=lambda k: jobs_dict[k]['rmse'])
        
        best_10 = sorted_job_ids[:10]
        worst_10 = sorted_job_ids[-10:][::-1]
        mid_idx = len(sorted_job_ids) // 2
        median_10 = sorted_job_ids[mid_idx - 5 : mid_idx + 5]

        generate_interactive_plot(best_10, jobs_dict, PLOTS_DIR / "best_10_jobs.html", "Top 10 Best Predictions")
        generate_interactive_plot(median_10, jobs_dict, PLOTS_DIR / "median_10_jobs.html", "Median 10 Average Predictions")
        generate_interactive_plot(worst_10, jobs_dict, PLOTS_DIR / "worst_10_jobs.html", "Bottom 10 Worst Predictions")

        print(f"[SUCCESS] Testing Diagnostics saved to: {PLOTS_DIR.name}/")
        
    except KeyboardInterrupt:
        print("\n[!] Testing interrupted by user (Ctrl+C).")
    except Exception as e:
        print(f"\n[!] Fatal Error during testing: {e}")
    finally:
        total_time = time.perf_counter() - total_start_time
        formatted_time = str(timedelta(seconds=int(total_time)))
        print(f"\n[*] Total Testing Time: {formatted_time}")
        print("=" * 70)

        if hasattr(sys.stdout, 'terminal'):
            sys.stdout.close()
            sys.stdout = sys.stdout.terminal

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'seaborn'